# Diagnosing and fixing two numerical problems in fracridge

## Executive summary

This notebook provides executable evidence that two defects in `fracridge` are
fixed. It calls the package as it exists: no monkeypatching and no replacement
of package internals. Local helper functions reproduce the two removed
algorithms so the corrected package can be measured against them on identical
data.

The two corrected defects are:

1. `FracRidgeRegressorCV` used to preprocess the complete dataset before
   cross-validation and then predict raw inputs with an intercept learned in
   centered coordinates. It now validates raw inputs and lets each inner
   estimator own the preprocessing for its own training fold.
2. `_do_svd` used to decompose `X.T @ X` for tall designs, squaring the
   condition number and producing inaccurate coefficients on nearly collinear
   data. It now computes a direct economy SVD of `X` and applies the
   singular-value tolerance before division.

Fraction `1.0` is used to isolate both defects from intentional ridge
shrinkage. Assertions make the numerical claims executable: the legacy
reproductions still fail, and the corrected package succeeds. The notebook is
committed without outputs so reviewers can reproduce the evidence locally.

### Legacy reproductions are local, not monkeypatches

Two helpers below, `fit_legacy_cv` and `legacy_normal_equations_coefficients`,
reimplement algorithms that no longer exist in the package. They exist only to
supply a measured baseline for the corrected code. Nothing here patches,
subclasses, or otherwise alters package internals, so every corrected number
comes from the installed package exactly as a user would call it.

`fit_legacy_cv` does call the base estimator's `_validate_input`, which is
still part of `FracRidgeRegressor`. Only its use *before* fold construction was
removed from `FracRidgeRegressorCV.fit`.

## Problem 1: cross-validation loses the intercept

### Expected invariant: affine OLS behavior

The deterministic data below obey `y = 3 + 2X` exactly. With the only candidate
fraction set to `1.0`, fractional ridge should reduce to the unregularized
solution: slope `2`, intercept `3`, maximum prediction error near machine
precision, and R-squared `1`. A constant prediction offset therefore cannot be
attributed to intended coefficient shrinkage.

### Legacy code path: preprocessing in the wrong coordinate system

The failure began in `FracRidgeRegressorCV.fit`. Its call to
`self._validate_input(X, y)` invoked `_preprocess_data`, centering the complete
dataset when `fit_intercept=True`. Those already-centered arrays were then
given to `GridSearchCV`, whose inner `FracRidgeRegressor` centered every fold
again.

After the final refit, the outer estimator copied `coef_` and `intercept_` from
`best_estimator_`. That intercept was expressed relative to centered inputs,
but the inherited `predict` method was later called with raw `X`. The
coordinate offset removed before cross-validation was never restored. Centering
before the fold split also let whole-dataset offsets influence every fold's
input.

`fit_legacy_cv` reproduces that exact sequence, so the cells below compare it
against the corrected estimator on one fixed shuffled split and a single
fraction.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, KFold

from fracridge.fracridge import (
    FracRidgeRegressor,
    FracRidgeRegressorCV,
    fracridge,
)

### Why the raw-data control isolates the fix

The control changes one boundary: raw `X` and `y` enter `GridSearchCV`.
Consequently, each training fold computes its own offsets exactly once, and the
final refit computes offsets from the complete raw training data exactly once.
The fitted estimator then predicts in the same coordinate system in which its
intercept was defined.

The control is independent of the package patch: it uses only
`FracRidgeRegressor` and `GridSearchCV`. If the corrected
`FracRidgeRegressorCV` reproduces the control's predictions while the legacy
reproduction fails on the same data and folds, preprocessing placement was the
isolated cause.

In [ ]:
def fit_legacy_cv(X, y, *, frac_grid, cv):
    """Reproduce the removed outer-preprocessing sequence locally."""
    preprocessor = FracRidgeRegressor(fit_intercept=True, jit=False)
    X_centered, y_centered, _, _, _ = preprocessor._validate_input(X, y)
    inner = FracRidgeRegressor(fit_intercept=True, jit=False)
    search = GridSearchCV(inner, {"fracs": np.asarray(frac_grid)}, cv=cv)
    search.fit(X_centered, y_centered)
    return search.best_estimator_


def fit_raw_data_control(X, y, *, frac_grid, cv):
    """Let a single intercept-fitting estimator center raw data once."""
    estimator = FracRidgeRegressor(fit_intercept=True, jit=False)
    search = GridSearchCV(estimator, {"fracs": np.asarray(frac_grid)}, cv=cv)
    return search.fit(X, y).best_estimator_


def summarize_affine_fit(estimator, X, y):
    """Collect the quantities that expose a coordinate-system mismatch."""
    prediction = estimator.predict(X)
    return {
        "prediction": prediction,
        "intercept": float(np.asarray(estimator.intercept_).squeeze()),
        "slope": float(np.asarray(estimator.coef_).squeeze()),
        "max_error": float(np.max(np.abs(prediction - y))),
        "r_squared": r2_score(y, prediction),
    }

In [ ]:
X_intercept = np.linspace(0.0, 1.0, 101)[:, np.newaxis]
y_intercept = 3.0 + 2.0 * X_intercept[:, 0]
cv = KFold(n_splits=5, shuffle=True, random_state=20260804)
frac_grid = np.array([1.0])

legacy_summary = summarize_affine_fit(
    fit_legacy_cv(X_intercept, y_intercept, frac_grid=frac_grid, cv=cv),
    X_intercept,
    y_intercept,
)

corrected_cv = FracRidgeRegressorCV(fit_intercept=True, cv=cv, jit=False)
corrected_cv.fit(X_intercept, y_intercept, frac_grid=frac_grid)
corrected_summary = summarize_affine_fit(corrected_cv, X_intercept, y_intercept)

control_summary = summarize_affine_fit(
    fit_raw_data_control(X_intercept, y_intercept, frac_grid=frac_grid, cv=cv),
    X_intercept,
    y_intercept,
)

for label, summary in (
    ("legacy", legacy_summary),
    ("corrected", corrected_summary),
    ("control", control_summary),
):
    print(
        f"{label:>10}: intercept={summary['intercept']:.12g} "
        f"slope={summary['slope']:.12g} "
        f"max_error={summary['max_error']:.12g} "
        f"R2={summary['r_squared']:.12g}"
    )

assert abs(legacy_summary["intercept"] - 3.0) > 2.5
assert legacy_summary["max_error"] > 2.5
assert np.isclose(corrected_summary["intercept"], 3.0, atol=1e-10)
assert np.isclose(corrected_summary["slope"], 2.0, atol=1e-10)
assert corrected_summary["max_error"] < 1e-10
assert np.isclose(corrected_summary["r_squared"], 1.0, atol=1e-12)
assert np.allclose(
    corrected_summary["prediction"], control_summary["prediction"], atol=1e-10
)

### Interpreting the legacy and corrected estimators

The legacy path keeps slope `2`, showing that it identified the relationship
between `X` and `y`. Its intercept is near zero and every prediction is
displaced by approximately `3`; the negative R-squared follows from that
displacement alone. This was a coordinate-system mismatch, not a failure to
estimate the slope.

The corrected estimator recovers intercept `3`, machine-precision prediction
error, R-squared `1`, and predictions that match the raw-data control. The
overlaid plot makes the contrast concrete: the legacy predictions have the
correct slope but the wrong vertical origin.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(X_intercept[:, 0], y_intercept, label="truth", linewidth=3)
ax.plot(
    X_intercept[:, 0],
    legacy_summary["prediction"],
    "--",
    label="legacy outer-preprocessing path",
)
ax.plot(
    X_intercept[:, 0],
    corrected_summary["prediction"],
    ":",
    label="corrected FracRidgeRegressorCV",
    linewidth=3,
)
ax.set(xlabel="X", ylabel="prediction", title="Intercept handling")
ax.legend()
fig.tight_layout()
plt.show()

### Problem 1 resolution

`FracRidgeRegressorCV.fit` now validates input shape with `validate_data`
without centering the complete dataset, passes raw validated inputs to
`GridSearchCV`, and copies the fitted state from `best_estimator_`. The
constructor also passes `fracs=None` to the base class instead of `self`.

Committed regression tests in `fracridge/tests/test_cross_validation.py` cover
the constructor state, nonzero-offset affine data for one and two targets, and
a `fit_intercept=False` control. They were observed failing before the
implementation changed.

## Problem 2: normal equations amplify numerical error

### Expected invariant: stable OLS at fraction 1.0

At fraction `1.0`, fractional ridge should reproduce a stable unregularized
least-squares solution. The comparisons below use `np.linalg.lstsq`, which
solves from `X` directly, as the numerical reference.

### Legacy code path: normal equations square conditioning

For a tall design, `_do_svd` used to decompose `X.T @ X` rather than `X`. For
the nonzero singular spectrum,

```text
cond(X.T @ X) = cond(X)^2
```

Squaring the condition number consumes roughly twice as many decimal digits of
precision. Errors in the recovered singular system were therefore present
before fracridge computed its shrinkage factors or rotated coefficients back to
the original basis.

The experiment below creates that failure in a controlled way by making two
columns progressively more collinear while holding every random draw fixed.
`legacy_normal_equations_coefficients` reproduces the removed fraction-`1.0`
coefficient path so the corrected package can be measured against it.

### Why the direct-SVD path solves the root cause

`_do_svd` now computes an economy SVD of `X` itself for every shape, projects
`y` into the left singular basis, and divides by singular values only where
they exceed `tol`. The condition number is never squared, and singular
directions at or below the tolerance are never divided.

The local `fracridge_direct_svd` control is retained as an independent
implementation of the same idea: it decomposes `X` directly and uses a
monotonic bisection, rather than the package's interpolation, to reach a
requested fraction. It accepts only a one-dimensional response; the production
implementation preserves vector, multi-target, tall, and wide shapes.

In [ ]:
def coefficient_norm_at_alpha(singular_values, ols_rotated, alpha):
    singular_values_sq = singular_values**2
    shrinkage = singular_values_sq / (singular_values_sq + alpha)
    return float(np.linalg.norm(shrinkage * ols_rotated))


def solve_fraction_alpha(singular_values, ols_rotated, fraction):
    if not 0.0 < fraction <= 1.0:
        raise ValueError("fraction must be in the interval (0, 1]")
    if fraction == 1.0:
        return 0.0

    target_norm = fraction * np.linalg.norm(ols_rotated)
    lower = 0.0
    upper = float(singular_values[0] ** 2)
    while coefficient_norm_at_alpha(singular_values, ols_rotated, upper) > target_norm:
        upper *= 2.0

    for _ in range(100):
        midpoint = (lower + upper) / 2.0
        midpoint_norm = coefficient_norm_at_alpha(
            singular_values, ols_rotated, midpoint
        )
        if midpoint_norm > target_norm:
            lower = midpoint
        else:
            upper = midpoint
    return (lower + upper) / 2.0


def direct_svd_components(X, y, tol):
    if np.ndim(y) != 1:
        raise ValueError("y must be one-dimensional")

    left, singular_values, right_transpose = np.linalg.svd(
        X, full_matrices=False
    )
    active = singular_values > tol
    if not np.any(active):
        raise ValueError("X has no singular values above tol")
    projected_y = left[:, active].T @ y
    ols_rotated = projected_y / singular_values[active]
    return singular_values[active], right_transpose[active], ols_rotated


def fracridge_direct_svd(X, y, fraction, *, tol=1e-10):
    singular_values, right_transpose, ols_rotated = direct_svd_components(
        X, y, tol
    )
    alpha = solve_fraction_alpha(singular_values, ols_rotated, fraction)
    singular_values_sq = singular_values**2
    shrinkage = singular_values_sq / (singular_values_sq + alpha)
    coefficients = right_transpose.T @ (shrinkage * ols_rotated)
    return coefficients, alpha


def legacy_normal_equations_coefficients(X, y):
    """Reproduce the removed tall fraction-1.0 normal-equations path."""
    _, squared_values, right_transpose = np.linalg.svd(X.T @ X)
    singular_values = np.sqrt(squared_values)
    rotated_y = (right_transpose @ (X.T @ y)) / singular_values
    return right_transpose.T @ (rotated_y / singular_values)

### Controlled experiment and reported metrics

Only `epsilon`, the perturbation separating the first two columns, changes
between rows. Smaller epsilon produces a larger `cond(X)`. For every design the
table reports:

- **legacy error:** coefficient disagreement between the reproduced `X.T @ X`
  path and `np.linalg.lstsq`;
- **package error:** the same disagreement for the corrected package at
  fraction `1.0`;
- **control error:** the same disagreement for the local direct-SVD control;
- **legacy residual** and **package residual:** the norms of `X @ coef - y`.

The coefficient errors test whether the solver recovered the stable parameter
vector. Residuals test only whether the fitted predictions reproduce this
particular response.

In [ ]:
def make_tall_design(base, perturbation, extras, epsilon):
    near_duplicate = base + epsilon * perturbation
    return np.column_stack((base, near_duplicate, extras))


rng = np.random.default_rng(20260804)
n_samples = 240
base_column = rng.normal(size=n_samples)
perturbation = rng.normal(size=n_samples)
extra_columns = rng.normal(size=(n_samples, 3))
true_coefficients = np.array([1.0, -1.0, 0.5, -0.25, 0.125])
epsilons = np.logspace(-4, -8, 5)
records = []

for epsilon in epsilons:
    design = make_tall_design(
        base_column, perturbation, extra_columns, epsilon
    )
    response = design @ true_coefficients
    least_squares = np.linalg.lstsq(design, response, rcond=None)[0]
    legacy_coefficients = legacy_normal_equations_coefficients(
        design, response
    )
    package_coefficients, _ = fracridge(
        design, response, fracs=1.0, jit=False
    )
    control_coefficients, _ = fracridge_direct_svd(
        design, response, fraction=1.0
    )
    denominator = np.linalg.norm(least_squares)
    records.append(
        {
            "epsilon": epsilon,
            "condition": np.linalg.cond(design),
            "legacy_error": np.linalg.norm(
                legacy_coefficients - least_squares
            )
            / denominator,
            "package_error": np.linalg.norm(
                package_coefficients - least_squares
            )
            / denominator,
            "control_error": np.linalg.norm(
                control_coefficients - least_squares
            )
            / denominator,
            "legacy_residual": np.linalg.norm(
                design @ legacy_coefficients - response
            ),
            "package_residual": np.linalg.norm(
                design @ package_coefficients - response
            ),
        }
    )

print(
    f"{'epsilon':>10} {'cond(X)':>14} {'legacy error':>15} "
    f"{'package error':>15} {'control error':>15} "
    f"{'legacy residual':>17} {'package residual':>18}"
)
for record in records:
    print(
        f"{record['epsilon']:10.1e} {record['condition']:14.6e} "
        f"{record['legacy_error']:15.6e} "
        f"{record['package_error']:15.6e} "
        f"{record['control_error']:15.6e} "
        f"{record['legacy_residual']:17.6e} "
        f"{record['package_residual']:18.6e}"
    )

worst_case = records[-1]
assert worst_case["legacy_error"] > 0.1
assert worst_case["package_error"] < 1e-6
assert worst_case["control_error"] < 1e-6
assert worst_case["package_error"] < worst_case["legacy_error"]

### Interpreting coefficient error and residual error

As conditioning worsens, the legacy coefficient error grows from numerical
noise to a material fraction of the least-squares coefficient norm. The
corrected package stays close to `np.linalg.lstsq` across the same designs and
tracks the independent direct-SVD control.

A small residual does not imply accurate coefficients here. Nearly collinear
columns permit substantially different coefficient vectors to produce similar
predictions, and the legacy residuals stay small even where its coefficients
are badly wrong. That is why coefficient error, rather than residual alone,
exposes the damage caused by forming `X.T @ X`.

The next check confirms that the corrected package is not merely an OLS
replacement. At requested fraction `0.5`, it must return a coefficient vector
whose norm is close to half the stable OLS norm, within the same interpolation
tolerance the package's own fraction tests use.

In [ ]:
fraction = 0.5
design = make_tall_design(
    base_column, perturbation, extra_columns, epsilons[-1]
)
response = design @ true_coefficients
ols_coefficients = np.linalg.lstsq(design, response, rcond=None)[0]
package_fractional, package_alpha = fracridge(
    design, response, fracs=fraction, jit=False
)
achieved_fraction = np.linalg.norm(package_fractional) / np.linalg.norm(
    ols_coefficients
)

print(f"requested fraction: {fraction:.12g}")
print(f"achieved fraction: {achieved_fraction:.12g}")
print(f"package alpha: {float(np.asarray(package_alpha).squeeze()):.12g}")

assert abs(achieved_fraction - fraction) < 0.01

The log-log plot summarizes the conditioning experiment. A stable method should
remain near the numerical floor as `cond(X)` increases; growth tied to
conditioning indicates that the decomposition itself is amplifying error.

In [ ]:
conditions = np.array([record["condition"] for record in records])
legacy_errors = np.array([record["legacy_error"] for record in records])
package_errors = np.array([record["package_error"] for record in records])
plot_floor = np.finfo(float).eps

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(conditions, legacy_errors, "o-", label="legacy X.T @ X path")
ax.loglog(
    conditions,
    np.maximum(package_errors, plot_floor),
    "o-",
    label="corrected package fracridge",
)
ax.set(
    xlabel="condition number of X",
    ylabel="relative coefficient error",
    title="Normal equations square the conditioning problem",
)
ax.legend()
fig.tight_layout()
plt.show()

### Problem 2 resolution

`_do_svd` computes one economy SVD of `X` for both tall and wide inputs and
applies the rank tolerance before dividing by singular values. `fracridge`
passes its `tol` through and uses the same active criterion for its warning and
its mask.

Committed regression tests in `fracridge/tests/test_numerical_stability.py`
cover fraction-`1.0` agreement with stable least squares for both JIT modes,
multi-target shapes and values, requested fractions below one, and a
rank-deficient tall design. They were observed failing before the
implementation changed.

## Pull request verification checklist

- [x] Focused intercept and numerical-stability regression tests were committed
      and observed failing before implementation.
- [x] CV preprocessing occurs within each estimator fit, not before fold
      construction.
- [x] A direct economy SVD of `X` replaces the normal-equations decomposition.
- [x] Tests cover intercept recovery, coefficient accuracy, achieved fraction,
      vector and multi-target shapes, and rank-deficient behavior.
- [x] The corrected wide-matrix OLS oracle was committed separately, before any
      new test or production change.
- [x] Ruff and the complete package suite pass, including doctests.

The legacy reproductions in this notebook are evidence, not package code. They
should be deleted if this notebook is ever repurposed as user-facing
documentation.